# YZM5110 — ISIC-DICM-17K: data audit and baseline preparation

This notebook starts with an **audit**, not a reported reproduction. Run it on Colab, inspect its printed columns and split diagnostics, and save the outputs before training. No numerical results are claimed here.

Source: [Ahammed et al., CVPRW 2025](https://openaccess.thecvf.com/content/CVPR2025W/MULA2025/html/Ahammed_Skin_Lesion_Classification_Using_Dermoscopic_Images_and_Clinical_Metadata_Insights_CVPRW_2025_paper.html). Official [ISIC-DICM-17K collection](https://api.isic-archive.com/doi/isic-dicm-17k/).

The paper used 15,098 training and 1,962 validation images, then evaluated on the ISIC 2020 test set through Kaggle. A new split of the downloadable bundle is **not** that published protocol. We must locate the original split IDs before claiming reproduction. The paper reports an ImageNet-pretrained EfficientNet-B3 multimodal AUC of 0.8851; this is an **authors' result**, not ours.


In [ ]:
from pathlib import Path
import platform, sys, time, json
import pandas as pd
import numpy as np
import torch

print({'python': sys.version.split()[0], 'pandas': pd.__version__,
       'numpy': np.__version__, 'torch': torch.__version__,
       'cuda': torch.cuda.is_available(),
       'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None})


## Download the official bundle

The official complete ZIP is about 6.9 GB. The following cell downloads it into Colab's temporary storage. It may need substantial free disk space when extracted. Do not commit images or raw metadata to the public repository. Individual source images have different licenses (CC-0, CC-BY, CC-BY-NC); follow the attribution and noncommercial limits shown by ISIC.


In [ ]:
import urllib.request, zipfile, shutil

DATA_DIR = Path('/content/isic-dicm-17k')
ZIP_PATH = Path('/content/isic-dicm-17k.zip')
URL = 'https://isic-archive.s3.amazonaws.com/dois/10.34970-233480/isic-dicm-17k.zip'
DATA_DIR.mkdir(parents=True, exist_ok=True)
if not ZIP_PATH.exists():
    urllib.request.urlretrieve(URL, ZIP_PATH)
print('ZIP bytes:', ZIP_PATH.stat().st_size)
print('Free GB:', round(shutil.disk_usage('/content').free / 1e9, 2))
with zipfile.ZipFile(ZIP_PATH) as archive:
    names = archive.namelist()
    print('Entries:', len(names), 'Examples:', names[:12])
    bad = [n for n in names if Path(n).is_absolute() or '..' in Path(n).parts]
    if bad:
        raise ValueError(f'Unsafe archive paths: {bad[:5]}')
    if not any(DATA_DIR.iterdir()):
        archive.extractall(DATA_DIR)


## Inspect actual files and metadata

Do not assume the downloaded CSV has the same column names as the paper's broader ISIC Archive table. Inspect its schema and missingness first.


In [ ]:
from collections import Counter

files = [p for p in DATA_DIR.rglob('*') if p.is_file()]
print('File count:', len(files))
print('Extensions:', Counter(p.suffix.lower() for p in files))
print('First paths:', [str(p.relative_to(DATA_DIR)) for p in files[:20]])
csv_paths = sorted(p for p in files if p.suffix.lower() == '.csv')
print('CSV files:', [str(p.relative_to(DATA_DIR)) for p in csv_paths])
for path in csv_paths:
    frame = pd.read_csv(path, low_memory=False)
    print('\n', path.name, 'shape:', frame.shape)
    print('columns:', list(frame.columns))
    print('missing:', frame.isna().sum().sort_values(ascending=False).to_dict())
    print(frame.head(3).to_string(index=False))


## Validate labels, image mapping, and split identifiers

Set the four column variables **after** seeing the printed schema. Never put diagnosis, diagnosis confirmation, or label-derived fields among clinical inputs. If patient IDs are missing, report that limitation; image-level splitting alone cannot prove patient independence.


In [ ]:
# Edit these only after inspecting the CSV above.
METADATA_CSV = None  # Example: csv_paths[0]
IMAGE_ID_COL = None
LABEL_COL = None
PATIENT_ID_COL = None
LESION_ID_COL = None

if METADATA_CSV is None or IMAGE_ID_COL is None or LABEL_COL is None:
    print('Set METADATA_CSV, IMAGE_ID_COL and LABEL_COL after auditing the real files.')
else:
    df = pd.read_csv(METADATA_CSV, low_memory=False)
    required = [IMAGE_ID_COL, LABEL_COL]
    assert all(c in df.columns for c in required)
    print('Labels:', df[LABEL_COL].value_counts(dropna=False).to_dict())
    print('Duplicate image IDs:', int(df[IMAGE_ID_COL].duplicated().sum()))
    for col in [PATIENT_ID_COL, LESION_ID_COL]:
        if col:
            assert col in df.columns, col
            print(col, 'missing:', int(df[col].isna().sum()),
                  'distinct:', df[col].nunique(dropna=True))
    image_paths = [p for p in files if p.suffix.lower() in {'.jpg', '.jpeg', '.png'}]
    image_by_stem = {p.stem: p for p in image_paths}
    ids = df[IMAGE_ID_COL].astype(str)
    print('Images found by filename stem:', int(ids.isin(image_by_stem).sum()), '/', len(ids))
    print('Unmatched IDs:', ids[~ids.isin(image_by_stem)].head(10).tolist())


## Compare the authors' published train/validation manifests

The authors' repository contains `train-set-metadata.csv` and `valid-set-metadata.csv`, with an explicit `class` column. Use these manifests instead of inferring the paper's binary labels from the official archive's `diagnosis_1` field. The official bundle's CSV has no `benign_malignant` field. This audit checks matching IDs, class balance, and patient/lesion overlap; inspect the results before training. The ISIC 2020 Kaggle test remains a separate evaluation.


In [ ]:
BASE = 'https://raw.githubusercontent.com/mmu-dermatology-research/isic-dicm-17k/main/'
train_manifest = pd.read_csv(BASE + 'train-set-metadata.csv', low_memory=False)
valid_manifest = pd.read_csv(BASE + 'valid-set-metadata.csv', low_memory=False)
archive_meta = pd.read_csv(DATA_DIR / 'metadata.csv', low_memory=False)

print('Manifest shapes:', train_manifest.shape, valid_manifest.shape)
print('Train columns:', list(train_manifest.columns))
print('Validation columns:', list(valid_manifest.columns))
for name, part in [('train', train_manifest), ('valid', valid_manifest)]:
    print(name, 'class counts:', part['class'].value_counts(dropna=False).to_dict())
    print(name, 'duplicate ISIC IDs:', int(part['isic_id'].duplicated().sum()))
    print(name, 'IDs absent from bundle CSV:', int((~part['isic_id'].isin(archive_meta['isic_id'])).sum()))
    for key in ['patient_id', 'lesion_id']:
        print(name, key, 'missing:', int(part[key].isna().sum()))

for key in ['isic_id', 'patient_id', 'lesion_id']:
    a = set(train_manifest[key].dropna().astype(str))
    b = set(valid_manifest[key].dropna().astype(str))
    print(key, 'train/valid overlap:', len(a & b))

combined = pd.concat([train_manifest[['isic_id', 'class']], valid_manifest[['isic_id', 'class']]])
print('Combined unique IDs:', combined['isic_id'].nunique(), 'of', len(archive_meta))
print('Combined class counts:', combined['class'].value_counts(dropna=False).to_dict())
print('Combined IDs absent from bundle CSV:', int((~combined['isic_id'].isin(archive_meta['isic_id'])).sum()))

# Compare published labels to archive diagnosis fields; do not treat these
# post-diagnosis fields as model inputs.
label_check = combined.merge(archive_meta[['isic_id', 'diagnosis_1', 'diagnosis_2']],
                             on='isic_id', how='left', validate='one_to_one')
print('class x diagnosis_1:')
print(pd.crosstab(label_check['class'], label_check['diagnosis_1'], dropna=False))


## Split gate

Before model training: locate the original validation image IDs in the authors' repository or supplemental files and identify how ISIC 2020 test evaluation can be repeated. For a separate internal experiment, split by patient ID where complete and verify zero overlap; when IDs are incomplete, investigate lesion IDs and duplicate images. Fit imputers and encoders **only on training data**. Keep the test set untouched while choosing a method.

Next implementation stage: image-only pretrained CNN, metadata-only MLP, late-fusion model, and one matched-condition intervention, with macro-F1, balanced accuracy, accuracy, ROC-AUC, confusion matrix, per-class scores, timing, and saved configurations. Record deviations from the paper explicitly.
